# Part 2 · How far L4 goes: identity authorisation without the L7 hop

In ambient, ztunnel enforces access policy at L4 on every hop, using each workload's certificate identity. Before you reach for an L7 waypoint, it is worth seeing how much you can already do at L4 and reduce the hop. 

Authorise on identity, on namespace, on conditions, deny selectively, and even tell apart two pods that share a ServiceAccount. 

The trust domain on `mesh1` is **`mesh1`** (a Helm value at install, not `cluster.local`), so identities read `spiffe://mesh1/ns/petshop/sa/<sa>`.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="800" height="300" rx="10" fill="#f8fafc"/><text x="400" y="22" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">What happens on an L4 connection: ztunnel proves and enforces identity</text><defs><marker id="za" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="zg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="zx" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#94a3b8"/></marker></defs><rect x="325" y="32" width="150" height="32" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/><text x="400" y="48" text-anchor="middle" font-size="10.5" font-weight="700" fill="#92400e">istiod  ·  the CA</text><text x="400" y="59" text-anchor="middle" font-size="8" fill="#92400e">mints and signs every SVID</text><line x1="360" y1="64" x2="256" y2="84" stroke="#94a3b8" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#zx)"/><line x1="440" y1="64" x2="544" y2="84" stroke="#94a3b8" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#zx)"/><text x="400" y="80" text-anchor="middle" font-size="8" fill="#94a3b8">issues certs</text><rect x="12" y="122" width="94" height="54" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="59" y="145" text-anchor="middle" font-size="11" font-weight="600" fill="#1e293b">storefront</text><text x="59" y="161" text-anchor="middle" font-size="8.5" fill="#475569">pod, no sidecar</text><text x="129" y="141" text-anchor="middle" font-size="8" fill="#64748b">plaintext</text><line x1="108" y1="149" x2="150" y2="149" stroke="#334155" stroke-width="1.6" marker-end="url(#za)"/><rect x="152" y="84" width="180" height="130" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/><text x="242" y="102" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ztunnel · node A</text><text x="242" y="117" text-anchor="middle" font-size="8.5" fill="#4338ca">presents the caller’s cert</text><rect x="168" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="173" y1="134" x2="191" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="173" y1="141" x2="191" y2="141" stroke="#e2e8f0"/><line x1="173" y1="146" x2="187" y2="146" stroke="#e2e8f0"/><circle cx="189" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="208" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="213" y1="134" x2="231" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="213" y1="141" x2="231" y2="141" stroke="#e2e8f0"/><line x1="213" y1="146" x2="227" y2="146" stroke="#e2e8f0"/><circle cx="229" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="248" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="253" y1="134" x2="271" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="253" y1="141" x2="271" y2="141" stroke="#e2e8f0"/><line x1="253" y1="146" x2="267" y2="146" stroke="#e2e8f0"/><circle cx="269" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="288" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="293" y1="134" x2="311" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="293" y1="141" x2="311" y2="141" stroke="#e2e8f0"/><line x1="293" y1="146" x2="307" y2="146" stroke="#e2e8f0"/><circle cx="309" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><text x="242" y="182" text-anchor="middle" font-size="8" fill="#4338ca">a leaf cert per local pod</text><text x="242" y="194" text-anchor="middle" font-size="7.5" fill="#4338ca">storefront · petstore · checkout · analytics</text><text x="242" y="207" text-anchor="middle" font-size="7.5" fill="#64748b">held by ztunnel, signed by istiod</text><text x="400" y="140" text-anchor="middle" font-size="10" font-weight="700" fill="#0f172a">mTLS · HBONE :15008</text><line x1="332" y1="154" x2="468" y2="154" stroke="#16a34a" stroke-width="3" marker-end="url(#zg)"/><text x="400" y="170" text-anchor="middle" font-size="8.5" fill="#166534">carries the caller identity</text><rect x="468" y="84" width="180" height="130" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/><text x="558" y="102" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ztunnel · node B</text><text x="558" y="117" text-anchor="middle" font-size="8.5" fill="#4338ca">checks the policy on that identity</text><rect x="484" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="489" y1="134" x2="507" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="489" y1="141" x2="507" y2="141" stroke="#e2e8f0"/><line x1="489" y1="146" x2="503" y2="146" stroke="#e2e8f0"/><circle cx="505" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="524" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="529" y1="134" x2="547" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="529" y1="141" x2="547" y2="141" stroke="#e2e8f0"/><line x1="529" y1="146" x2="543" y2="146" stroke="#e2e8f0"/><circle cx="545" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="564" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="569" y1="134" x2="587" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="569" y1="141" x2="587" y2="141" stroke="#e2e8f0"/><line x1="569" y1="146" x2="583" y2="146" stroke="#e2e8f0"/><circle cx="585" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><rect x="604" y="124" width="28" height="40" rx="3" fill="#ffffff" stroke="#94a3b8"/><line x1="609" y1="134" x2="627" y2="134" stroke="#c7d2fe" stroke-width="2"/><line x1="609" y1="141" x2="627" y2="141" stroke="#e2e8f0"/><line x1="609" y1="146" x2="623" y2="146" stroke="#e2e8f0"/><circle cx="625" cy="156" r="4" fill="#fde68a" stroke="#d97706"/><text x="558" y="182" text-anchor="middle" font-size="8" fill="#4338ca">a leaf cert per local pod</text><text x="558" y="194" text-anchor="middle" font-size="7.5" fill="#4338ca">storefront · petstore · checkout · analytics</text><text x="558" y="207" text-anchor="middle" font-size="7.5" fill="#64748b">held by ztunnel, signed by istiod</text><text x="671" y="141" text-anchor="middle" font-size="8" fill="#64748b">allow</text><line x1="650" y1="149" x2="692" y2="149" stroke="#16a34a" stroke-width="1.8" marker-end="url(#zg)"/><rect x="694" y="122" width="94" height="54" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="741" y="153" text-anchor="middle" font-size="11" font-weight="600" fill="#1e293b">petstore</text><text x="400" y="248" text-anchor="middle" font-size="10.5" fill="#334155">identity in the cert: spiffe://mesh1/ns/petshop/sa/storefront  ·  ALLOW if it matches the policy, else DENY (fail closed)</text><text x="400" y="276" text-anchor="middle" font-size="10" fill="#64748b">Every node’s ztunnel holds a leaf cert per local pod (signed by istiod), proves the caller over mTLS, and enforces policy at L4.</text></svg></div>

Each client curls `petstore:8080/pets` every 2 seconds and prints the HTTP code. The observe cells read the client's last log line, which is why each policy step sleeps briefly first. A denied client shows `000` (connection reset), an allowed one `200`.

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect** first, then **Reset** for a clean slate, then the steps top to bottom.
> It needs `./demo-scripts/setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

## Connect · run this first

Sets the contexts, the Solo `istioctl` build and your licence env, and confirms the platform is up. **Safe to re-run**: it only sets env, it does not touch the cluster. (Reloaded the notebook? Run just this to get your env back.)

In [ ]:
# Connect: context, image/chart versions, licences. Safe to re-run (env only).
[ -d vision-demo-2026 ] && cd vision-demo-2026 || :
# clear strays (dev servers, other labs' stale port-forwards) off this demo's local ports;
# Docker publishes and this suite's own kubectl forwards survive
./demo-scripts/free-ports.sh 8091
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel >/dev/null 2>&1 && echo "ambient mesh: up on ${CTX#kind-}" || echo "mesh not found on ${CTX#kind-}: run ./demo-scripts/setup.sh (or ./demo-scripts/wake.sh after a sleep)"

### Consoles / URLs

Watch traffic in the **Gloo UI** service graph. Open it once and leave it running (I open the URL in the Cursor browser):

```
./demo-scripts/consoles.sh
```

| Console | URL |
|---|---|
| Gloo UI (service graph) | http://localhost:8091 |

**Graph tips:** tick **mesh1** and the **petshop** namespace, then Graph Settings (gear) → **Idle Nodes OFF**, Traffic = **Last 1 min**, and give each change ~15-30s to show up.

Note: at L4 a denial is a dropped connection, so a blocked caller shows as an **edge going quiet**, not a red edge (that changes at L7 in Part 3).


## Reset · clean slate

Resets the whole demo to square one (deletes the demo namespaces, reverts ztunnel), leaving the platform up so there is no rebuild. Run it **before a fresh run**, or skip it if you just reloaded and only needed the env from Connect above.

In [ ]:
# Reset the whole demo to a clean slate: deletes the demo namespaces and reverts
# ztunnel, but leaves the platform up. Run this to start fresh, NOT just for env.
bash "$(git rev-parse --show-toplevel)/vision-demo-2026/demo-scripts/reset.sh"

## 2.1 · Deploy the petshop app

**What we're doing:** deploy the petshop (an API plus four client workloads) into one namespace and enrol it in the mesh.

**How:** a single `istio.io/dataplane-mode=ambient` label on the namespace, no restarts and no sidecars.

**What you'll see:** the five pods running, ready for the identity walk-through.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

kubectl --context $CTX apply -f demo-scripts/yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods

## 2.2 · The certificate is the identity

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="740" height="300" rx="10" fill="#f8fafc"/>
  <text x="370" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Five pods, four identities: the certificate is the workload's identity</text>
  <defs>
    <marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker>
    <marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker>
  </defs>
  <text x="99" y="50" text-anchor="middle" font-size="10" font-weight="600" fill="#64748b">pods</text>
  <text x="594" y="50" text-anchor="middle" font-size="10" font-weight="600" fill="#64748b">identity (SPIFFE SVID = SA leaf cert)</text>
  <rect x="24" y="58" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="99" y="80" text-anchor="middle" font-size="11" fill="#1e293b">analytics</text>
  <rect x="24" y="102" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="99" y="124" text-anchor="middle" font-size="11" fill="#1e293b">checkout-blue</text>
  <rect x="24" y="146" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="99" y="168" text-anchor="middle" font-size="11" fill="#1e293b">checkout-green</text>
  <rect x="24" y="190" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="99" y="212" text-anchor="middle" font-size="11" fill="#1e293b">petstore</text>
  <rect x="24" y="234" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="99" y="256" text-anchor="middle" font-size="11" fill="#1e293b">storefront</text>
  <rect x="468" y="58" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/>
  <text x="594" y="79" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/analytics</text>
  <rect x="468" y="120" width="252" height="42" rx="7" fill="#fef3c7" stroke="#d97706" stroke-width="1.8"/>
  <text x="594" y="137" text-anchor="middle" font-size="9.5" font-weight="600" fill="#7c2d12">spiffe://mesh1/ns/petshop/sa/checkout</text>
  <text x="594" y="152" text-anchor="middle" font-size="8.5" fill="#92400e">one SVID for BOTH checkout pods</text>
  <rect x="468" y="190" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/>
  <text x="594" y="211" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/petstore</text>
  <rect x="468" y="234" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/>
  <text x="594" y="255" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/storefront</text>
  <line x1="174" y1="75" x2="466" y2="75" stroke="#334155" stroke-width="1.6" marker-end="url(#d)"/>
  <line x1="174" y1="119" x2="466" y2="137" stroke="#d97706" stroke-width="2" marker-end="url(#a)"/>
  <line x1="174" y1="163" x2="466" y2="145" stroke="#d97706" stroke-width="2" marker-end="url(#a)"/>
  <line x1="174" y1="207" x2="466" y2="207" stroke="#334155" stroke-width="1.6" marker-end="url(#d)"/>
  <line x1="174" y1="251" x2="466" y2="251" stroke="#334155" stroke-width="1.6" marker-end="url(#d)"/>
  <text x="370" y="290" text-anchor="middle" font-size="11" fill="#64748b">Five pods, four ServiceAccounts. checkout-blue and checkout-green share sa/checkout, so they present the SAME cert: one identity to the mesh.</text>
</svg></div>

**What we're doing:** show that a workload's identity is its mTLS certificate, issued from its ServiceAccount, and that this is what policy matches on.

**How:** list the leaf certificates ztunnel holds, next to the pods.

**What you'll see:** five pods but only **four** certificates. `checkout-blue` and `checkout-green` never appear by name because they share `sa/checkout`, so ztunnel presents one certificate for both. To the mesh they are the same caller. That is the ceiling §2.5 runs into and §2.7 removes.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"

## 2.3 · Authorise on identity, at L4

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="700" height="260" rx="10" fill="#f8fafc"/> <text x="350" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">ztunnel decides on identity: allow storefront, deny the rest</text> <defs> <marker id="lg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="lr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- callers on the left --> <rect x="24" y="52" width="150" height="34" rx="7" fill="#dcfce7" stroke="#16a34a"/> <text x="99" y="74" text-anchor="middle" font-size="11.5" fill="#14532d">storefront</text> <rect x="24" y="98" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="120" text-anchor="middle" font-size="11.5" fill="#7f1d1d">analytics</text> <rect x="24" y="144" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="166" text-anchor="middle" font-size="11.5" fill="#7f1d1d">checkout-blue</text> <rect x="24" y="190" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="212" text-anchor="middle" font-size="11.5" fill="#7f1d1d">checkout-green</text> <!-- ztunnel gate in the middle --> <rect x="300" y="86" width="90" height="90" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/> <text x="345" y="126" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ztunnel</text> <text x="345" y="142" text-anchor="middle" font-size="9.5" fill="#4338ca">L4 authz</text> <!-- arrows callers -> ztunnel --> <line x1="174" y1="69"  x2="300" y2="110" stroke="#16a34a" stroke-width="2" marker-end="url(#lg)"/> <line x1="174" y1="115" x2="300" y2="128" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <line x1="174" y1="161" x2="300" y2="140" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <line x1="174" y1="207" x2="300" y2="158" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <!-- petstore on the right, only the allowed one gets through --> <line x1="390" y1="120" x2="500" y2="120" stroke="#16a34a" stroke-width="2" marker-end="url(#lg)"/> <text x="445" y="112" text-anchor="middle" font-size="10" fill="#166534">200</text> <rect x="500" y="98" width="176" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="588" y="125" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <text x="350" y="246" text-anchor="middle" font-size="11" fill="#64748b">One ALLOW on the storefront identity. ztunnel fails closed: everything unnamed is denied, no waypoint.</text> </svg></div>

**What we're doing:** allow only the `storefront` workload to reach `petstore`, and let ztunnel deny everything else, with no waypoint and no app change.

**How:** one `AuthorizationPolicy` that selects `petstore` and permits the `storefront` identity. ztunnel fails closed: once any ALLOW selects a workload, everything not named is denied. Principals use the `mesh1` trust domain, so a `cluster.local/...` principal here would match nothing.

**What you'll see:** `storefront` gets `200`, while `analytics` and both `checkout` pods get `000` (connection refused), decided purely on identity.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s; read its latest result and show the verdict
echo "${CYN}${BLD}  ══ who can reach petstore?  (only the storefront identity is allowed) ══${RST}"
for d in storefront analytics checkout-blue checkout-green; do
  code=$(kubectl --context $CTX -n petshop logs deploy/$d --tail=1 2>/dev/null | awk '{print $NF}')
  [ "$code" = "200" ] && v="${GRN}${BLD}200  ✓ ALLOW${RST}" || v="${RED}${BLD}${code:-000}  ✗ DENY${RST}"
  printf "    %-16s %s\n" "$d" "$v"
done

**Watching in the Gloo UI and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry**. A workload denied at L4 never completes a request, so it should draw nothing, but a pre-policy 200 can linger as a stale edge. Turn **"Idle Nodes" OFF** (Graph Settings) so only live traffic draws, and set Traffic to **Last 1 min**. The definitive, per-connection allow/deny verdicts are in ztunnel's access logs, next.

## 2.4 · Read the L4 access logs

**What we're doing:** show that the allow/deny decisions are auditable, tagged with the caller's identity, at L4 with no waypoint.

**How:** read ztunnel's access log, which records every connection with the peer SPIFFE identities and the outcome.

**What you'll see:** `storefront` as **ALLOW** and `analytics` as **DENY**, each line carrying its `src.identity`.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'

## 2.5 · The limit of service-account identity

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 220" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="220" rx="10" fill="#f8fafc"/>
  <text x="360" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">The ceiling: two pods share one ServiceAccount, so L4 cannot separate them</text>
  <defs>
    <marker id="g5" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker>
  </defs>
  <rect x="24" y="56" width="180" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="114" y="76" text-anchor="middle" font-size="11.5" font-weight="600" fill="#1e293b">checkout-blue</text>
  <text x="114" y="91" text-anchor="middle" font-size="9" fill="#475569">identity: sa/checkout</text>
  <rect x="24" y="120" width="180" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="114" y="140" text-anchor="middle" font-size="11.5" font-weight="600" fill="#1e293b">checkout-green</text>
  <text x="114" y="155" text-anchor="middle" font-size="9" fill="#475569">identity: sa/checkout</text>
  <line x1="204" y1="79" x2="298" y2="100" stroke="#16a34a" stroke-width="2" marker-end="url(#g5)"/>
  <line x1="204" y1="143" x2="298" y2="122" stroke="#16a34a" stroke-width="2" marker-end="url(#g5)"/>
  <rect x="300" y="78" width="160" height="66" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/>
  <text x="380" y="100" text-anchor="middle" font-size="10.5" font-weight="700" fill="#312e81">AuthorizationPolicy</text>
  <text x="380" y="115" text-anchor="middle" font-size="9.5" fill="#4338ca">action: ALLOW</text>
  <text x="380" y="129" text-anchor="middle" font-size="9.5" fill="#4338ca">principal: sa/checkout</text>
  <line x1="460" y1="111" x2="538" y2="111" stroke="#16a34a" stroke-width="2" marker-end="url(#g5)"/>
  <text x="499" y="103" text-anchor="middle" font-size="9" fill="#166534">both ✓</text>
  <rect x="540" y="89" width="160" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="620" y="115" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">petstore</text>
  <text x="360" y="184" text-anchor="middle" font-size="11" fill="#b91c1c">No L4 rule can admit blue but block green: they present the same certificate.</text>
  <text x="360" y="204" text-anchor="middle" font-size="11" fill="#64748b">§2.7 removes this ceiling with a per-pod claim baked into each certificate.</text>
</svg></div>

**What we're doing:** show the one thing this model cannot do: tell apart two pods that share a ServiceAccount.

**How:** add `sa/checkout` to the allowed set and call from both checkout pods.

**What you'll see:** both `checkout-blue` and `checkout-green` get in, and there is no L4 rule that admits one and blocks the other. They share `sa/checkout`, present the same certificate, and to ztunnel they are the same caller.

Why it matters: the everyday version is worse than blue/green. Any pod that never sets a ServiceAccount runs as the namespace `default`, so an ALLOW written for "the payments app" quietly admits every workload in that namespace. The two fixes are to give every workload its own ServiceAccount (as this app does) and, where pods genuinely share one, to close the gap with **workload claims** in §2.7.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
echo "${CYN}${BLD}  ══ now sa/checkout is allowed too  ·  BOTH checkout pods get in ══${RST}"
for d in storefront analytics checkout-blue checkout-green; do
  code=$(kubectl --context $CTX -n petshop logs deploy/$d --tail=1 2>/dev/null | awk '{print $NF}')
  [ "$code" = "200" ] && v="${GRN}${BLD}200  ✓ ALLOW${RST}" || v="${RED}${BLD}${code:-000}  ✗ DENY${RST}"
  printf "    %-16s %s\n" "$d" "$v"
done
echo "    checkout-blue and checkout-green share sa/checkout — no L4 rule separates them"

## 2.6 · What else L4 can decide: namespace, conditions, DENY

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="740" height="250" rx="10" fill="#f8fafc"/>
  <text x="370" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">L4 decides on more than identity: namespace, conditions, and DENY overrides ALLOW</text>
  <defs>
    <marker id="g6" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker>
    <marker id="r6" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker>
  </defs>
  <rect x="20" y="56" width="180" height="40" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="110" y="72" text-anchor="middle" font-size="11" font-weight="600" fill="#1e293b">storefront</text>
  <text x="110" y="86" text-anchor="middle" font-size="8.5" fill="#475569">ns: petshop</text>
  <rect x="20" y="110" width="180" height="40" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="110" y="126" text-anchor="middle" font-size="11" font-weight="600" fill="#1e293b">warehouse-svc</text>
  <text x="110" y="140" text-anchor="middle" font-size="8.5" fill="#475569">ns: warehouse</text>
  <rect x="20" y="164" width="180" height="40" rx="7" fill="#dbeafe" stroke="#60a5fa"/>
  <text x="110" y="180" text-anchor="middle" font-size="11" font-weight="600" fill="#1e293b">analytics</text>
  <text x="110" y="194" text-anchor="middle" font-size="8.5" fill="#475569">ns: petshop</text>
  <line x1="200" y1="76" x2="298" y2="112" stroke="#16a34a" stroke-width="2" marker-end="url(#g6)"/>
  <text x="250" y="88" text-anchor="middle" font-size="8.5" fill="#166534">✓ namespace</text>
  <line x1="200" y1="130" x2="298" y2="130" stroke="#16a34a" stroke-width="2" marker-end="url(#g6)"/>
  <text x="250" y="122" text-anchor="middle" font-size="8.5" fill="#166534">✓ namespace</text>
  <line x1="200" y1="184" x2="298" y2="150" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#r6)"/>
  <text x="250" y="182" text-anchor="middle" font-size="8.5" fill="#991b1b">✗ DENY wins</text>
  <rect x="300" y="88" width="188" height="84" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/>
  <text x="394" y="112" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ztunnel · L4</text>
  <text x="394" y="129" text-anchor="middle" font-size="9" fill="#4338ca">evaluates DENY, then ALLOW</text>
  <text x="394" y="143" text-anchor="middle" font-size="9" fill="#4338ca">on identity, namespace, port,</text>
  <text x="394" y="155" text-anchor="middle" font-size="9" fill="#4338ca">and CEL when-conditions</text>
  <line x1="488" y1="130" x2="558" y2="130" stroke="#16a34a" stroke-width="2" marker-end="url(#g6)"/>
  <rect x="560" y="108" width="160" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="640" y="134" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">petstore</text>
  <text x="370" y="230" text-anchor="middle" font-size="11" fill="#64748b">One ALLOW can admit a whole namespace; a DENY on an identity overrides it. All at L4, no waypoint.</text>
</svg></div>

**What we're doing:** show that identity is not the only thing ztunnel can authorise on. It also decides on source namespace, source IP, destination port and SNI, directly or in a CEL `when` clause, and a `DENY` always overrides an `ALLOW`.

**How:** add a second caller in its own `warehouse` namespace, then walk it through allow, block and an explicit deny.

**What you'll see:** the warehouse caller allowed when the policy admits its namespace, then blocked when we narrow it, and finally an explicit `DENY` overriding an `ALLOW`. You can watch it appear and disappear in the Gloo UI Graph.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
# A second client, in ANOTHER namespace — its identity is spiffe://mesh1/ns/warehouse/sa/warehouse-svc
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s

# ── verify: warehouse client is up and its identity is in the warehouse trust domain ──
kubectl --context $CTX -n warehouse get pods
echo "${RED}no ALLOW admits it yet, so it is denied for now:${RST}"
sleep 6; kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1 | sed "s/: 000/: ${RED}${BLD}000${RST}/; s/: 403/: ${RED}${BLD}403${RST}/"

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 158" style="width:100%;max-width:900px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="158" rx="10" fill="#f8fafc"/><text x="360" y="22" text-anchor="middle" font-size="13.5" font-weight="700" fill="#0f172a">Allow by namespace: one rule admits a whole namespace</text><defs><marker id="ag" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="ar" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker></defs><rect x="20" y="44" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="65" text-anchor="middle" font-size="10" fill="#1e293b">storefront (ns petshop)</text><rect x="20" y="92" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="113" text-anchor="middle" font-size="10" fill="#1e293b">warehouse-svc (ns warehouse)</text><line x1="200" y1="61" x2="276" y2="78" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><line x1="200" y1="109" x2="276" y2="98" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><text x="238" y="72" text-anchor="middle" font-size="9" fill="#166534">✓</text><text x="238" y="120" text-anchor="middle" font-size="9" fill="#166534">✓</text><rect x="278" y="54" width="212" height="62" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/><text x="384" y="74" text-anchor="middle" font-size="10" font-weight="700" fill="#312e81">AuthorizationPolicy · ALLOW</text><text x="384" y="90" text-anchor="middle" font-size="8.5" fill="#4338ca">when source.namespace in</text><text x="384" y="103" text-anchor="middle" font-size="8.5" fill="#4338ca">{ petshop, warehouse }</text><line x1="490" y1="85" x2="552" y2="85" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><rect x="554" y="67" width="146" height="36" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="627" y="90" text-anchor="middle" font-size="11" font-weight="600" fill="#14532d">petstore</text><text x="360" y="146" text-anchor="middle" font-size="10" fill="#64748b">Keys on source.namespace (a CEL when-condition), not identity. Both namespaces reach petstore.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE ALLOW admits callers from TWO namespaces — decided by a CEL `when` on source.namespace
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop", "warehouse"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"
# both 200 — refresh the Graph: warehouse appears, serving petstore

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 158" style="width:100%;max-width:900px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="158" rx="10" fill="#f8fafc"/><text x="360" y="22" text-anchor="middle" font-size="13.5" font-weight="700" fill="#0f172a">Narrow the rule: drop warehouse, it fails closed</text><defs><marker id="ag" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="ar" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker></defs><rect x="20" y="44" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="65" text-anchor="middle" font-size="10" fill="#1e293b">storefront (ns petshop)</text><rect x="20" y="92" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="113" text-anchor="middle" font-size="10" fill="#1e293b">warehouse-svc (ns warehouse)</text><line x1="200" y1="61" x2="276" y2="78" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><line x1="200" y1="109" x2="276" y2="98" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#ar)"/><text x="238" y="72" text-anchor="middle" font-size="9" fill="#166534">✓</text><text x="238" y="120" text-anchor="middle" font-size="9" fill="#991b1b">✗</text><rect x="278" y="54" width="212" height="62" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/><text x="384" y="74" text-anchor="middle" font-size="10" font-weight="700" fill="#312e81">AuthorizationPolicy · ALLOW</text><text x="384" y="90" text-anchor="middle" font-size="8.5" fill="#4338ca">when source.namespace ==</text><text x="384" y="103" text-anchor="middle" font-size="8.5" fill="#4338ca">petshop</text><line x1="490" y1="85" x2="552" y2="85" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><rect x="554" y="67" width="146" height="36" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="627" y="90" text-anchor="middle" font-size="11" font-weight="600" fill="#14532d">petstore</text><text x="360" y="146" text-anchor="middle" font-size="10" fill="#64748b">An ALLOW exists but none matches warehouse, so it is denied (fail-closed). No explicit deny needed.</text></svg></div>

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# Now BLOCK it — same policy, one value removed. warehouse fails closed.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"

# the deny as ztunnel logged it — FAIL-CLOSED: allow policies exist, none matched
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep warehouse | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 158" style="width:100%;max-width:900px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="158" rx="10" fill="#f8fafc"/><text x="360" y="22" text-anchor="middle" font-size="13.5" font-weight="700" fill="#0f172a">DENY overrides ALLOW: block one identity inside an allowed namespace</text><defs><marker id="ag" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="ar" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker></defs><rect x="20" y="44" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="65" text-anchor="middle" font-size="10" fill="#1e293b">storefront (ns petshop)</text><rect x="20" y="92" width="180" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="110" y="113" text-anchor="middle" font-size="10" fill="#1e293b">analytics (ns petshop)</text><line x1="200" y1="61" x2="276" y2="78" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><line x1="200" y1="109" x2="276" y2="98" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#ar)"/><text x="238" y="72" text-anchor="middle" font-size="9" fill="#166534">✓</text><text x="238" y="120" text-anchor="middle" font-size="9" fill="#991b1b">✗</text><rect x="278" y="54" width="212" height="62" rx="9" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.8"/><text x="384" y="74" text-anchor="middle" font-size="10" font-weight="700" fill="#312e81">ztunnel evaluates</text><text x="384" y="90" text-anchor="middle" font-size="8.5" fill="#4338ca">DENY, then ALLOW</text><text x="384" y="103" text-anchor="middle" font-size="8.5" fill="#4338ca">DENY sa/analytics wins</text><line x1="490" y1="85" x2="552" y2="85" stroke="#16a34a" stroke-width="2" marker-end="url(#ag)"/><rect x="554" y="67" width="146" height="36" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="627" y="90" text-anchor="middle" font-size="11" font-weight="600" fill="#14532d">petstore</text><text x="360" y="146" text-anchor="middle" font-size="10" fill="#64748b">analytics is in the allowed namespace, but a DENY on its identity wins. ztunnel checks DENY before ALLOW.</text></svg></div>

**DENY overrides ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW admits it) but a `DENY` on its identity blocks it anyway, a clean carve-out with no change to the broader policy. From the client both denials are the same `000`; the difference is in ztunnel's log: the warehouse deny read `allow policies exist, but none allowed` (fail-closed); this one reads `explicitly denied by: petshop/l4-deny-analytics`; the log **names the policy** that fired.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done

for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep analytics | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# reset the L4-surface policies before closing the gap
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found

# ── verify: only the SA-wide allow-storefront + allow-checkout remain ──
kubectl --context $CTX -n petshop get authorizationpolicy

## 2.7 · Close the gap with workload claims

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="700" height="250" rx="10" fill="#f8fafc"/> <text x="350" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Workload claims: same ServiceAccount, told apart by a signed claim</text> <defs> <marker id="cg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="cr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- blue pod (gold) --> <rect x="24" y="58" width="210" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="129" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">checkout-blue</text> <text x="129" y="96" text-anchor="middle" font-size="10" fill="#166534">sa/checkout · cert claim tier=gold</text> <!-- green pod (silver) --> <rect x="24" y="140" width="210" height="52" rx="8" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/> <text x="129" y="161" text-anchor="middle" font-size="12" font-weight="600" fill="#7f1d1d">checkout-green</text> <text x="129" y="178" text-anchor="middle" font-size="10" fill="#991b1b">sa/checkout · cert claim tier=silver</text> <!-- policy gate --> <rect x="300" y="88" width="110" height="74" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/> <text x="355" y="120" text-anchor="middle" font-size="11" font-weight="700" fill="#312e81">CEL when</text> <text x="355" y="137" text-anchor="middle" font-size="9.5" fill="#4338ca">tier == "gold"</text> <line x1="234" y1="84"  x2="300" y2="112" stroke="#16a34a" stroke-width="2" marker-end="url(#cg)"/> <line x1="234" y1="166" x2="300" y2="138" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#cr)"/> <!-- petstore --> <line x1="410" y1="125" x2="500" y2="125" stroke="#16a34a" stroke-width="2" marker-end="url(#cg)"/> <text x="455" y="117" text-anchor="middle" font-size="10" fill="#166534">200</text> <rect x="500" y="103" width="176" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="588" y="130" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <text x="350" y="228" text-anchor="middle" font-size="11" fill="#64748b">Both share sa/checkout, so §2.5 could not separate them. The per-pod cert claim can, still at L4.</text> </svg></div>

**What we're doing:** solve the §2.5 problem, telling apart two pods on one ServiceAccount, still at L4 and still with no waypoint.

**How:** turn on `ENABLE_WORKLOAD_CLAIMS`. ztunnel then requests a certificate **per pod** instead of per ServiceAccount, istiod embeds signed claims in each one at issuance, and an `AuthorizationPolicy` matches those claims with CEL. On the 1.30 line this is one Helm value on ztunnel, same chart and version, all the multicluster values kept.

**What you'll see (over the next few cells):** every pod now holding its own certificate, `checkout-blue` carrying `tier: gold` and `checkout-green` carrying `tier: silver` in that certificate, and a claims-based policy that finally lets blue in and keeps green out.

> **Turn on workload claims.** One helm value flips ztunnel from one cert per ServiceAccount to one per pod. This cell sets it and confirms the flag is live.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE new value: ENABLE_WORKLOAD_CLAIMS. Everything else identical to the standup.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s

# ── verify: the flag is live on ztunnel ──
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel \
  -o jsonpath='{.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].name}={.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].value}{"\n"}'

With claims turned on, ztunnel mints one certificate **per pod**: `checkout-blue` and `checkout-green` no longer share a leaf. They still present the same `sa/checkout` SPIFFE URI, and there is no `tier` claim yet, that gets stamped in when we annotate the pods next (contrast [§2.2](#22)):

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="740" height="300" rx="10" fill="#f8fafc"/><text x="370" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">After workload claims: five pods, five per-pod certificates (checkout no longer shared)</text><defs><marker id="ad" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="aa" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker></defs><text x="99" y="50" text-anchor="middle" font-size="10" font-weight="600" fill="#64748b">pods</text><text x="594" y="50" text-anchor="middle" font-size="10" font-weight="600" fill="#64748b">identity (per-pod SVID = leaf cert)</text><rect x="24" y="58" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="99" y="80" text-anchor="middle" font-size="11" fill="#1e293b">analytics</text><rect x="24" y="102" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="99" y="124" text-anchor="middle" font-size="11" fill="#1e293b">checkout-blue</text><rect x="24" y="146" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="99" y="168" text-anchor="middle" font-size="11" fill="#1e293b">checkout-green</text><rect x="24" y="190" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="99" y="212" text-anchor="middle" font-size="11" fill="#1e293b">petstore</text><rect x="24" y="234" width="150" height="34" rx="7" fill="#dbeafe" stroke="#60a5fa"/><text x="99" y="256" text-anchor="middle" font-size="11" fill="#1e293b">storefront</text><rect x="468" y="58" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/><text x="594" y="79" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/analytics</text><rect x="468" y="99" width="252" height="40" rx="7" fill="#fffbeb" stroke="#f59e0b" stroke-width="1.8"/><text x="594" y="116" text-anchor="middle" font-size="9" fill="#312e81">spiffe://mesh1/ns/petshop/sa/checkout</text><text x="594" y="131" text-anchor="middle" font-size="8" fill="#92400e">own per-pod leaf · checkout-blue</text><rect x="468" y="143" width="252" height="40" rx="7" fill="#fffbeb" stroke="#f59e0b" stroke-width="1.8"/><text x="594" y="160" text-anchor="middle" font-size="9" fill="#312e81">spiffe://mesh1/ns/petshop/sa/checkout</text><text x="594" y="175" text-anchor="middle" font-size="8" fill="#92400e">own per-pod leaf · checkout-green</text><rect x="468" y="190" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/><text x="594" y="211" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/petstore</text><rect x="468" y="234" width="252" height="34" rx="7" fill="#e0e7ff" stroke="#6366f1"/><text x="594" y="255" text-anchor="middle" font-size="9.5" fill="#312e81">spiffe://mesh1/ns/petshop/sa/storefront</text><line x1="174" y1="75" x2="466" y2="75" stroke="#334155" stroke-width="1.6" marker-end="url(#ad)"/><line x1="174" y1="119" x2="466" y2="119" stroke="#d97706" stroke-width="2" marker-end="url(#aa)"/><line x1="174" y1="163" x2="466" y2="163" stroke="#d97706" stroke-width="2" marker-end="url(#aa)"/><line x1="174" y1="207" x2="466" y2="207" stroke="#334155" stroke-width="1.6" marker-end="url(#ad)"/><line x1="174" y1="251" x2="466" y2="251" stroke="#334155" stroke-width="1.6" marker-end="url(#ad)"/><text x="370" y="288" text-anchor="middle" font-size="10.5" fill="#64748b">Both checkout pods now hold their own leaf (same SPIFFE URI). The signed tier claim is stamped in next, when we annotate the pods.</text></svg></div>

> **Look at the certs now.** In §2.2 ztunnel held four (one per ServiceAccount). With claims on it holds one per pod, so checkout-blue and checkout-green each get their own certificate.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# every workload now holds a PER-POD cert (contrast §2.2: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"

**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.

> **Tag the two pods.** blue = gold, green = silver, one annotation each. istiod bakes that claim into each pod's certificate at issuance. The app is untouched.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5

# ── verify: the tier annotation is on each pod (istiod bakes it into the cert) ──
kubectl --context $CTX -n petshop get pod -l app=checkout \
  -o custom-columns=POD:.metadata.name,TIER:'.metadata.annotations.solo\.io\.security-claims/tier'

**What that certificate looks like.** Here is checkout-blue's leaf certificate: the standard X.509 fields, plus the signed workload claims istiod baked in (`tier = gold`). checkout-green's is identical except the pod name and `tier = silver`.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 500" style="width:100%;max-width:860px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="500" rx="10" fill="#eef2f6"/>
  <rect x="12" y="12" width="696" height="476" rx="12" fill="#ffffff" stroke="#cbd5e1"/>
  <circle cx="48" cy="52" r="22" fill="#fef3c7" stroke="#d97706" stroke-width="2"/>
  <circle cx="48" cy="52" r="15" fill="none" stroke="#d97706" stroke-width="1"/>
  <text x="48" y="56" text-anchor="middle" font-size="10" font-weight="700" fill="#92400e">CA</text>
  <text x="84" y="42" font-size="15" font-weight="700" fill="#0f172a">checkout-blue leaf certificate</text>
  <text x="84" y="61" font-size="11" fill="#475569">Issued by: CN=mesh1 Intermediate CA  (O=Solo Demo)</text>
  <text x="84" y="77" font-size="11" fill="#475569">Valid about 24h (short-lived, auto-rotated by istiod)</text>
  <text x="84" y="97" font-size="11.5" font-weight="600" fill="#16a34a">✓ This certificate is valid</text>
  <line x1="24" y1="112" x2="696" y2="112" stroke="#e2e8f0" stroke-width="1"/>
  <text x="28" y="136" font-size="12" font-weight="700" fill="#334155">Details</text>
  <text x="250" y="162" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Subject (SAN, URI)</text>
  <text x="264" y="162" font-size="11" fill="#0f172a">spiffe://mesh1/ns/petshop/sa/checkout</text>
  <text x="250" y="186" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Issuer</text>
  <text x="264" y="186" font-size="11" fill="#0f172a">O=Solo Demo, CN=mesh1 Intermediate CA</text>
  <text x="250" y="210" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Signature Algorithm</text>
  <text x="264" y="210" font-size="11" fill="#0f172a">sha256WithRSAEncryption</text>
  <text x="250" y="234" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Not Before</text>
  <text x="264" y="234" font-size="11" fill="#0f172a">Jul 24 08:23:12 2026 GMT</text>
  <text x="250" y="258" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Not After</text>
  <text x="264" y="258" font-size="11" fill="#0f172a">Jul 25 08:25:12 2026 GMT  (about 24h)</text>
  <text x="250" y="282" text-anchor="end" font-size="11" font-weight="600" fill="#334155">Serial Number</text>
  <text x="264" y="282" font-size="11" fill="#0f172a">22f5a54741d901b474a4589717b3136d</text>
  <rect x="20" y="300" width="680" height="176" rx="8" fill="#fffbeb" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="34" y="320" font-size="10.5" font-weight="700" fill="#92400e">Workload claims  ·  otherName OID 1.3.6.1.4.1.65865.1.1  ·  base64url JSON, signed by istiod</text>
  <text x="250" y="346" text-anchor="end" font-size="11" font-weight="600" fill="#7c2d12">iss</text>
  <text x="264" y="346" font-size="11" fill="#0f172a">https://istiod.istio-system.svc.mesh1</text>
  <text x="250" y="368" text-anchor="end" font-size="11" font-weight="600" fill="#7c2d12">istio.io / trust_domain</text>
  <text x="264" y="368" font-size="11" fill="#0f172a">mesh1</text>
  <text x="250" y="390" text-anchor="end" font-size="11" font-weight="600" fill="#7c2d12">workload / name</text>
  <text x="264" y="390" font-size="11" fill="#0f172a">checkout</text>
  <text x="250" y="412" text-anchor="end" font-size="11" font-weight="600" fill="#7c2d12">workload / namespace</text>
  <text x="264" y="412" font-size="11" fill="#0f172a">petshop</text>
  <text x="250" y="434" text-anchor="end" font-size="11" font-weight="600" fill="#7c2d12">workload / pod</text>
  <text x="264" y="434" font-size="11" fill="#0f172a">checkout-blue-5c8fbf858b-rg6ll</text>
  <text x="250" y="458" text-anchor="end" font-size="11" font-weight="700" fill="#7c2d12">solo.io security-claims / tier</text>
  <rect x="264" y="446" width="60" height="18" rx="9" fill="#dcfce7" stroke="#16a34a"/>
  <text x="294" y="459" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">gold</text>
  <text x="360" y="484" text-anchor="middle" font-size="10.5" fill="#64748b">checkout-green's certificate is identical, except workload/pod and security-claims/tier = silver.</text>
</svg></div>

> **Read the real certificate.** Pull each pod's leaf cert off ztunnel and decode it: same SPIFFE URI on both, but blue carries tier=gold and green tier=silver, signed by istiod.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# Pull the REAL leaf certificate ztunnel presents for each checkout pod and read
# the claim off it. The claim rides in an otherName SAN under Solo's OID
# (1.3.6.1.4.1.65865.1.1); the payload is base64url JSON, signed by istiod.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem
  uri=$(openssl x509 -in /tmp/$pod.pem -noout -text | grep -oE 'URI:spiffe://[^,]+' | sed 's/URI://')
  tier=$(openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); d=json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))); print(d.get("solo.io",{}).get("security-claims",{}).get("tier","?"))' 2>/dev/null)
  echo "  ══ $pod  ·  real leaf certificate from ztunnel ══"
  printf "    %-14s %s\n" "SPIFFE URI:" "$uri"
  printf "    %-14s %s\n" "signed claim:" "tier = $tier   (issued + signed by istiod)"
  echo
done
echo
echo "  ══ full leaf certificate for checkout-blue (openssl x509 -text) ══"
openssl x509 -in /tmp/checkout-blue.pem -noout -text
echo
echo "  Same SPIFFE URI on both (…/sa/checkout) — the signed tier claim is what tells them apart."


<details>
<summary><strong>Click to expand: the leaf certificate that carries the signed claim</strong></summary>

istiod issues a **per-pod** X.509 leaf certificate to ztunnel over xDS, and the `tier` claim rides inside it, in an `otherName` SAN under Solo's OID `1.3.6.1.4.1.65865.1.1`. Pulled from ztunnel and decoded with `openssl`, `checkout-blue`'s certificate:

```
X509v3 Subject Alternative Name: critical
    URI:spiffe://mesh1/ns/petshop/sa/checkout,
    othername: 1.3.6.1.4.1.65865.1.1:eyJpc3MiOiJodHRwczovL2lzdGlvZC5p...   (base64url JSON, truncated)

issuer = O=Solo Demo, CN=mesh1 Intermediate CA
notAfter ~ 24h leaf
```

Base64url-decode that `otherName` payload and you get the signed claim set:

```json
{
  "iss": "https://istiod.istio-system.svc.mesh1",
  "sub": "spiffe://mesh1/ns/petshop/sa/checkout",
  "iat": 1784881512, "exp": 1784967912,
  "istio.io": {
    "trust_domain": "mesh1",
    "workload": { "name": "checkout", "namespace": "petshop", "pod": "checkout-blue-5c8fbf858b-rg6ll" }
  },
  "solo.io": { "security-claims": { "tier": "gold" } }
}
```

</details>

Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. 

Now authorize on it: swap the SA-wide checkout ALLOW for a claims-scoped one — the same `when` CEL shape as §2.6, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). 

Fail-closed: a workload with no claim never matches the ALLOW.

> **Authorize on the claim.** Allow only tier=gold. blue gets in, green is denied, same ServiceAccount, told apart by the signed claim. This is the thing §2.5 could not do.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
GRN=$'\e[32m'; RED=$'\e[31m'; YEL=$'\e[33m'; BLD=$'\e[1m'; CYN=$'\e[36m'; RST=$'\e[0m'
kubectl --context $CTX -n petshop delete authorizationpolicy allow-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - mesh1/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge; wait, then test
echo "  waiting ~25s for the per-pod certs and the policy to converge..."
sleep 25
B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore:8080/ 2>/dev/null)
G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore:8080/ 2>/dev/null)
echo
[ "${B:-000}" = "200" ] && echo "checkout-blue  (tier gold)   -> ${GRN}${BLD}${B}${RST}" || echo "checkout-blue  (tier gold)   -> ${RED}${BLD}${B:-000}${RST}"
[ "${G:-000}" = "200" ] && echo "checkout-green (tier silver) -> ${GRN}${BLD}${G}${RST}   (same sa/checkout, told apart by its cert claim)" || echo "checkout-green (tier silver) -> ${RED}${BLD}${G:-000}${RST}   (same sa/checkout, told apart by its cert claim)"

`checkout-blue -> 200`, `checkout-green -> denied`, same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §2.5 could not do.

That closes out the L4 story. Everything so far, identity, namespace, conditions, a deny, and now per-pod claims, was enforced by ztunnel at L4 with **no proxy in the path**. Part 3 picks up the things that genuinely need L7.

## Tear down and move on

Finished with this part? Run the cell below to start cleaning up, then switch to the next lab and talk it through while the cluster resets in the background. It deletes the demo namespaces and reverts ztunnel, but leaves the platform (clusters, ambient mesh, peering, agentgateway, Gloo UI, Keycloak) up, so the next lab needs no rebuild. It is the same reset the next lab's setup cell runs, so it is safe to run anytime.

In [ ]:
# Tear down this part while you switch to the next lab: deletes the demo namespaces
# and reverts ztunnel, but leaves the platform up. Safe to run anytime.
bash "$(git rev-parse --show-toplevel)/vision-demo-2026/demo-scripts/reset.sh"